In [1]:
%matplotlib inline
#
from multifunbrain import *
from multifunbrain.notebook import *
#
if Path.cwd().name != 'multifun-brain':
    path_root = Path.cwd().parent
    os.chdir(path_root)
#
path_data =  path_root / 'data'
path_fig = path_data / 'figures' /'25-11-05' / 'freq-user'
#
path_ts = path_data / 'atlas_timecourses'
path_corrmat = path_data / 'correlation_matrices'
path_freq_user = path_corrmat / 'freq-user'
#

In [2]:
# Fetch the Schaefer 2018 atlas.
atlas = datasets.fetch_atlas_schaefer_2018(
    n_rois=100, data_dir=path_data
)
atlas_filename = atlas.maps
atlas_img = image.load_img(atlas_filename)
atlas_data = atlas_img.get_fdata()
#
rois = atlas.labels
rois_label_map = {i: ll for i, ll in enumerate(rois)}
roi_indices = np.unique(list(rois_label_map.keys()))
# Load MNI152 template for the background.
template = load_mni152_template()

[get_dataset_dir] Dataset found in /home/opisthofulax/Documents/UniPa/projects/multifun-brain/data/schaefer_2018


In [3]:
# Find all .pkl files in nested subfolders (user/contrast/*.pkl)
files = list(path_freq_user.glob('**/*.pkl'))

# Build dictionary with user and contrast information
# File structure: path_freq_user / user_folder / contrast_folder / file.pkl
# File name: sub-XXXXXXXX_Schaefer2018_100Parcels_17Networks_kwXXX.ts.1D_sX.pkl
files_dict = {}
for file_path in files:
    # Extract user folder (e.g., '00') and contrast folder (e.g., '0', '1', '2')
    user_folder = file_path.parent.parent.name  # e.g., '00'
    contrast_folder = file_path.parent.name      # e.g., '0'
    
    # Extract subject ID from filename (e.g., 'sub-00027599')
    file_name = file_path.name
    if file_name.startswith('sub-'):
        subject_id = file_name.split('_')[0]  # 'sub-00027599'
    else:
        subject_id = f'user_{user_folder}'
    
    # Create label from filename parts
    parts = file_name.replace('.ts.1D', '').replace('.pkl', '').split('_')
    # Extract the last 3 meaningful parts (method, filter, band)
    label_suffix = '_'.join(parts[-3:])
    
    # Create composite key: subject_contrast_label
    label_key = f"{subject_id}_{contrast_folder}_{label_suffix}"
    
    files_dict[label_key] = {
        'path': file_path,
        'subject_id': subject_id,
        'user_folder': user_folder,
        'contrast_folder': contrast_folder,
        'label_suffix': label_suffix
    }

# Get sorted list of labels for iteration
labels_name = sorted(files_dict.keys())

print(f"Found {len(files_dict)} correlation matrices across users and contrasts")
print(f"\nExample labels:")
for label in list(labels_name)[:3]:
    info = files_dict[label]
    print(f"  {label}")
    print(f"    → User: {info['subject_id']}, Contrast: {info['contrast_folder']}")

Found 234 correlation matrices across users and contrasts

Example labels:
  sub-00027599_0_17Networks_kwCBF4D_s4
    → User: sub-00027599, Contrast: 0
  sub-00027599_0_17Networks_kwCBF4D_s5
    → User: sub-00027599, Contrast: 0
  sub-00027599_0_17Networks_kwCBF4D_sstar
    → User: sub-00027599, Contrast: 0


In [4]:
corr_matrices = {}
for label, file_info in files_dict.items():
    file = file_info['path']
    with open(file, 'rb') as f:
        matrix = pk.load(f)
    
    # Clean matrix: remove NaN and symmetrize
    matrix_no_nan = np.nan_to_num(matrix, nan=0.0)
    matrix_clean = (matrix_no_nan + matrix_no_nan.T) / 2
    
    # Marchenko-Pastur parameters
    n_channels, time_steps = matrix_clean.shape[0], 488  # Adjust time_steps if needed
    gamma = time_steps / n_channels
    lambda_min = (1 - np.sqrt(1/gamma))**2
    lambda_max = (1 + np.sqrt(1/gamma))**2
    
    # MP-based correlation matrix cleaning
    evals_orig, evecs_orig = np.linalg.eigh(matrix_clean)
    sort_idx = np.argsort(evals_orig)[::-1]
    evals_orig, evecs_orig = evals_orig[sort_idx], evecs_orig[:, sort_idx]
    
    # Identify signal vs noise eigenvectors based on MP bounds
    signal_mask = (evals_orig < lambda_min) | (evals_orig > lambda_max)
    n_signal, n_noise = np.sum(signal_mask), np.sum(~signal_mask)
    
    # Project onto signal subspace and convert back to correlation matrix
    signal_projector = evecs_orig[:, signal_mask] @ evecs_orig[:, signal_mask].T
    matrix_signal_proj = signal_projector @ matrix_clean @ signal_projector
    
    # Normalize to correlation form (cov2corr transformation)
    diag_elements = np.diag(matrix_signal_proj).copy()
    diag_elements[diag_elements <= 0] = 1e-10
    D_inv = np.diag(1.0 / np.sqrt(diag_elements))
    matrix_mp_cleaned = D_inv @ matrix_signal_proj @ D_inv
    
    # Ensure valid correlation matrix
    matrix_mp_cleaned = (matrix_mp_cleaned + matrix_mp_cleaned.T) / 2
    np.fill_diagonal(matrix_mp_cleaned, 1.0)
    
    # Store cleaned matrix
    corr_matrices[label] = matrix_mp_cleaned
    
    print(f"{label}: Removed {n_noise-np.sum((np.linalg.eigvals(matrix_mp_cleaned) >= lambda_min) & (np.linalg.eigvals(matrix_mp_cleaned) <= lambda_max))} noise eigenvalues")

sub-00027599_3_bold_s5_120s: Removed -14 noise eigenvalues
sub-00027599_3_bold_s4_120s: Removed 3 noise eigenvalues
sub-00027599_3_kwoptcomMIRDenoised_bold_s4: Removed 27 noise eigenvalues
sub-00027599_3_kwoptcomMIRDenoised_bold_sstar: Removed 32 noise eigenvalues
sub-00027599_3_bold_sstar_120s: Removed 22 noise eigenvalues
sub-00027599_3_kwoptcomMIRDenoised_bold_s5: Removed 14 noise eigenvalues
sub-00027599_0_17Networks_kwCBF4D_s5: Removed 12 noise eigenvalues
sub-00027599_0_17Networks_kwCBF4D_s4: Removed 17 noise eigenvalues
sub-00027599_0_17Networks_kwCBF4D_sstar: Removed -67 noise eigenvalues
sub-00027599_1_kwfcurN_Vaso_s5: Removed 22 noise eigenvalues
sub-00027599_1_kwfcurN_Vaso_s4: Removed 32 noise eigenvalues
sub-00027599_1_kwfcurN_Vaso_sstar: Removed 29 noise eigenvalues
sub-00027599_2_kwfurN_Bold_sstar: Removed 23 noise eigenvalues
sub-00027599_2_Bold_sstar_120s: Removed 18 noise eigenvalues
sub-00027599_2_Bold_s4_120s: Removed 9 noise eigenvalues
sub-00027599_2_kwfurN_Bold_s5

In [ ]:
# Comprehensive analysis for each correlation matrix
for fn in labels_name:
    # Extract user and contrast information
    file_info = files_dict[fn]
    subject_id = file_info['subject_id']
    contrast_folder = file_info['contrast_folder']
    user_folder = file_info['user_folder']
    
    # Create output directory: path_fig / subject_id / contrast_folder
    output_dir = path_fig / subject_id / contrast_folder
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Define all expected output files
    file_01 = output_dir / f'{fn}_01_correlation_matrix_and_network.pdf'
    file_02 = output_dir / f'{fn}_02_thresholding_network_entropy.pdf'
    file_03 = output_dir / f'{fn}_03_clustering_network_dendro_brain.pdf'
    file_04 = output_dir / f'{fn}_04_sankey_diagram.html'
    file_05 = output_dir / f'{fn}_05_multithreshold_brain_partitions.pdf'
    file_06_json = output_dir / f'{fn}_06_clustering_hierarchy.json'
    file_06_pkl = output_dir / f'{fn}_06_clustering_hierarchy.pkl'
    
    # Check which files exist
    skip_01 = file_01.exists()
    skip_02 = file_02.exists()
    skip_03 = file_03.exists()
    skip_04 = file_04.exists()
    skip_05 = file_05.exists()
    skip_06 = file_06_json.exists() and file_06_pkl.exists()
    
    all_exist = skip_01 and skip_02 and skip_03 and skip_04 and skip_05 and skip_06
    
    if all_exist:
        print(f"\n{'='*60}")
        print(f"SKIPPING (already processed): {fn}")
        print(f"  User: {subject_id}, Contrast: {contrast_folder}")
        print(f"  All output files exist (7 files)")
        print(f"{'='*60}")
        continue
    
    print(f"\n{'='*60}")
    print(f"Processing: {fn}")
    print(f"  User: {subject_id}, Contrast: {contrast_folder}")
    print(f"  Output: {output_dir}")
    print(f"  Files to generate: {sum([not skip_01, not skip_02, not skip_03, not skip_04, not skip_05, not skip_06])}/7")
    print(f"{'='*60}")
    
    # ========== 1. CORRELATION MATRIX AND INITIAL NETWORK ==========
    if not skip_01:
        try:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
            
            # Plot correlation matrix
            im = ax1.imshow(corr_matrices[fn], cmap='coolwarm', vmin=-1, vmax=1)
            ax1.set_title(f'Correlation Matrix: {fn}', fontsize=12)
            plt.colorbar(im, ax=ax1, shrink=0.8)
            
            # Plot initial network
            G_init, _ = get_giant_component_leftoff(nx.from_numpy_array(corr_matrices[fn]))
            G_init = nx.convert_node_labels_to_integers(G_init)
            edge_weights_init = [G_init[u][v].get('weight', 0) for u, v in G_init.edges()]
            
            if len(edge_weights_init) == 0:
                ax2.text(0.5, 0.5, 'No edges in graph', ha='center', va='center')
                ax2.set_title('Initial Network (no edges)', fontsize=12)
            else:
                norm_init = TwoSlopeNorm(vmin=min(edge_weights_init), vcenter=0, vmax=max(edge_weights_init))
                cmap_init = LinearSegmentedColormap.from_list(
                    'red_transparent_blue',
                    [(0.0, (1, 0, 0, 1)), (0.5, (0, 0, 0, .01)), (1.0, (0, 0, 1, 1))], N=256
                )
                edge_colors_init = [cmap_init(norm_init(w)) for w in edge_weights_init]
                nx.draw(G_init, ax=ax2, edge_color=edge_colors_init, node_size=10)
                sm_init = plt.cm.ScalarMappable(cmap=cmap_init, norm=norm_init)
                sm_init.set_array(edge_weights_init)
                plt.colorbar(sm_init, ax=ax2, label='Edge Weight', shrink=0.8)
                ax2.set_title('Initial Network (no threshold)', fontsize=12)
            
            plt.tight_layout()
            plt.savefig(file_01, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"  ✓ Generated: {file_01.name}")
        except Exception as e:
            plt.close('all')
            print(f"  ✗ Failed: {file_01.name} - {str(e)}")
    else:
        print(f"  ⊙ Skipped (exists): {file_01.name}")
    
    # ========== 2. THRESHOLDING PROCEDURE ==========
    # Always need to compute graph properties for later steps (even if figure exists)
    try:
        G, lost_nodes_0 = get_giant_component_leftoff(nx.from_numpy_array(corr_matrices[fn]))
        G = nx.convert_node_labels_to_integers(G)
        
        A_old = nx.to_numpy_array(G)
        A_new = np.where(A_old < 0, 0, A_old)
        G_pos = nx.from_numpy_array(A_new)
        Th, Einf, Pinf = compute_threshold_stats_fast(G_pos)
        Th, jump = find_threshold_jumps(G_pos)
        
        sel_thresh = Th[jump[0]] if len(jump) > 0 else Th[len(Th)//2]
        
        A_prime = nx.to_numpy_array(G_pos)
        A_thresh = np.where(A_prime < sel_thresh, 0, A_prime)
        G_pos, lost_nodes_1 = get_giant_component_leftoff(nx.from_numpy_array(A_thresh))
        L_pos, spectrum = graph_laplacian_and_spectrum(G_pos)
        Sm1, dS, VarL, tau = entropy(spectrum)
        
        if not skip_02:
            fig, ax = plt.subplots(ncols=3, figsize=(16, 4))
            
            # Thresholding plot
            ax[0].plot(Th, Pinf, 'h', label='P∞')
            ax[0].plot(Th, Einf, label='E∞')
            ax[0].axvline(sel_thresh, ls='--', color='red', label=f'Selected={sel_thresh:.3f}')
            ax[0].set_xlabel('Threshold')
            ax[0].set_ylabel('Percolation/Efficiency')
            ax[0].set_title('Threshold Selection')
            ax[0].legend()
            ax[0].grid(True, alpha=0.3)
            
            # Thresholded network
            pos_net = nx.spring_layout(G_pos, k=0.15, seed=42)
            nx.draw(G_pos, pos=pos_net, ax=ax[1], node_size=30)
            ax[1].set_title(f'Thresholded Network ({len(G_pos.nodes())} nodes)')
            
            # Entropy and specific heat
            ax[2].plot(tau, Sm1, label='Entropy S(τ)')
            ax[2].plot(tau[1:], dS, label='Specific Heat dS/dτ')
            ax[2].set_xscale('log')
            ax[2].set_xlabel('Temperature τ')
            ax[2].set_ylabel('S / dS')
            ax[2].set_title('Entropy Analysis')
            ax[2].legend()
            ax[2].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.savefig(file_02, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"  ✓ Generated: {file_02.name}")
        else:
            print(f"  ⊙ Skipped (exists): {file_02.name}")
    except Exception as e:
        if not skip_02:
            plt.close('all')
            print(f"  ✗ Failed: {file_02.name} - {str(e)}")
        print(f"  ✗ Error in thresholding procedure: {str(e)}")
        print(f"  → Skipping remaining analysis for {fn}")
        continue
    
    # ========== 3. CLUSTERING: NETWORK, DENDROGRAM, BRAIN ==========
    # Always need to compute clustering for later steps (even if figure exists)
    try:
        tau_opt = 1/max(spectrum)
        rho_m = lambda tau: rho_matrix(tau, L_pos)
        dists = symmetrized_inverse_distance(tau_opt, rho_m)
        linkage_matrix, label_list, tmax = compute_normalized_linkage(dists, G_pos, method='ward')
        tmin = linkage_matrix[::, 2][0] - 0.5*linkage_matrix[::, 2][0]
        threshold_opt, *_ = compute_optimal_threshold(linkage_matrix, scaling_factor=0.95)
        
        # Use a reasonable threshold for visualization
        threshold_viz = max(threshold_opt, 1e-7)
        optimal_clusters = fcluster(linkage_matrix, t=threshold_viz, criterion='distance')
        
        if not skip_03:
            # Set color palette
            n_colors = max(optimal_clusters)
            colors = plt.cm.tab20c(np.linspace(0, 1, n_colors))
            colors = [plt.matplotlib.colors.rgb2hex(c) for c in colors]
            set_link_color_palette(colors)
            
            fig = plt.figure(figsize=(14, 6))
            gs = gridspec.GridSpec(4, 6, figure=fig, hspace=0.3, wspace=0.3)
            
            # Dendrogram
            ax_dendro = fig.add_subplot(gs[0:2, 3:6])
            dendro = dendrogram(linkage_matrix, ax=ax_dendro, color_threshold=threshold_viz,
                                above_threshold_color='k', orientation='top', labels=label_list)
            ax_dendro.set_yscale('log')
            ax_dendro.axhline(threshold_viz, ls='--', color='red')
            ax_dendro.set_ylim(tmin, 1)
            ax_dendro.set_title(f'Dendrogram (threshold={threshold_viz:.2e})')
            
            # Graph with cluster colors
            ax_graph = fig.add_subplot(gs[0:4, 0:3])
            leaf_label_colors = {label: color for label, color in zip(dendro['ivl'], dendro['leaves_color_list'])}
            node_colors = [leaf_label_colors[label] for _, label in enumerate(label_list)]
            edge_weights = [G_pos[u][v].get('weight', 0) for u, v in G_pos.edges()]
            min_w, max_w = min(edge_weights), max(edge_weights)
            widths = [0.1 + (abs(w) - abs(min_w)) / (abs(max_w) - abs(min_w)) * (3 - 0.1) if max_w != min_w else 3 for w in edge_weights]
            pos_clust = nx.spring_layout(G_pos, seed=42, k=0.15)
            nx.draw(G_pos, pos=pos_clust, ax=ax_graph, width=widths, node_size=50, node_color=node_colors)
            ax_graph.set_title(f'Clustered Network ({n_colors} clusters)')
            
            # Brain projection
            ax_nilearn = fig.add_subplot(gs[2:4, 3:6])
            indices_to_insert = sorted(lost_nodes_0 + lost_nodes_1)
            node_colors_all = node_colors.copy()
            for idx in reversed(indices_to_insert):
                node_colors_all.insert(idx, 'white')
            cmap_brain = ListedColormap(node_colors_all)
            plotting.plot_stat_map(atlas_img, bg_img=template, axes=ax_nilearn, cmap=cmap_brain, colorbar=False)
            ax_nilearn.set_title('Brain Partition')
            
            plt.savefig(file_03, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"  ✓ Generated: {file_03.name}")
        else:
            print(f"  ⊙ Skipped (exists): {file_03.name}")
        
        # Store indices_to_insert for later use
        indices_to_insert = sorted(lost_nodes_0 + lost_nodes_1)
    except Exception as e:
        if not skip_03:
            plt.close('all')
            print(f"  ✗ Failed: {file_03.name} - {str(e)}")
        print(f"  ✗ Error in clustering procedure: {str(e)}")
        print(f"  → Skipping remaining analysis for {fn}")
        continue
    
    # ========== 4. SANKEY DIAGRAM AND MULTI-THRESHOLD BRAIN VIEWS ==========
    # Always need to compute thresholds and cluster assignments for section 5 (even if figures exist)
    try:
        # Target cluster numbers (from fine-grained to coarse)
        target_n_clusters = [100, 50, 25, 12, 6, 3]
        
        # Generate candidate thresholds
        candidate_thresholds = np.logspace(np.log10(tmin), np.log10(1), 200)
        
        # Find thresholds that produce cluster numbers closest to targets
        thresholds_sankey = []
        for target_n in target_n_clusters:
            best_thresh = None
            best_diff = float('inf')
            
            for t in candidate_thresholds:
                clusters = fcluster(linkage_matrix, t=t, criterion='distance')
                n_clust = len(np.unique(clusters))
                diff = abs(n_clust - target_n)
                
                # Prefer thresholds that produce the target number or slightly fewer clusters
                if diff < best_diff:
                    best_diff = diff
                    best_thresh = t
            
            if best_thresh is not None:
                thresholds_sankey.append(best_thresh)
        
        # Remove duplicates while preserving order (in case multiple targets map to same threshold)
        seen = set()
        thresholds_sankey_unique = []
        for t in thresholds_sankey:
            if t not in seen:
                seen.add(t)
                thresholds_sankey_unique.append(t)
        thresholds_sankey = np.array(thresholds_sankey_unique)
        
        # Get cluster assignments for each threshold
        cluster_assignments_sankey = {}
        actual_n_clusters = []
        for i, thresh in enumerate(thresholds_sankey):
            clusters = fcluster(linkage_matrix, t=thresh, criterion='distance')
            cluster_assignments_sankey[i] = clusters
            actual_n_clusters.append(len(np.unique(clusters)))
        
        print(f"  - Target clusters: {target_n_clusters}")
        print(f"  - Actual clusters: {actual_n_clusters}")
        print(f"  - Thresholds: {[f'{t:.2e}' for t in thresholds_sankey]}")
        
        if not skip_04 or not skip_05:
            # Generate consistent colors
            cluster_colors_dict_sankey = {}
            for level in range(len(thresholds_sankey)):
                unique_clusters = np.unique(cluster_assignments_sankey[level])
                n_clusters = len(unique_clusters)
                colors_level = plt.cm.tab20(np.linspace(0, 1, n_clusters))
                cluster_colors_dict_sankey[level] = {}
                for idx, cluster in enumerate(unique_clusters):
                    cluster_colors_dict_sankey[level][cluster] = plt.matplotlib.colors.rgb2hex(colors_level[idx])
        
        if not skip_04:
            try:
                # Build Sankey diagram data
                sources, targets, values, labels_sankey, node_colors_sankey = [], [], [], [], []
                
                for level in range(len(thresholds_sankey)):
                    unique_clusters = np.unique(cluster_assignments_sankey[level])
                    for cluster in unique_clusters:
                        labels_sankey.append(f"L{level}_C{cluster}")
                        node_colors_sankey.append(cluster_colors_dict_sankey[level][cluster])
                
                for level in range(len(thresholds_sankey) - 1):
                    current_clusters = cluster_assignments_sankey[level]
                    next_clusters = cluster_assignments_sankey[level + 1]
                    
                    for node_idx in range(len(current_clusters)):
                        curr_cluster = current_clusters[node_idx]
                        next_cluster = next_clusters[node_idx]
                        
                        source_idx = labels_sankey.index(f"L{level}_C{curr_cluster}")
                        target_idx = labels_sankey.index(f"L{level+1}_C{next_cluster}")
                        
                        try:
                            flow_idx = sources.index(source_idx)
                            if targets[flow_idx] == target_idx:
                                values[flow_idx] += 1
                                continue
                        except ValueError:
                            pass
                        
                        sources.append(source_idx)
                        targets.append(target_idx)
                        values.append(1)
                
                # Create Sankey diagram
                fig_sankey = go.Figure(data=[go.Sankey(
                    node=dict(pad=15, thickness=20, line=dict(color="black", width=0.5),
                              label=labels_sankey, color=node_colors_sankey),
                    link=dict(source=sources, target=targets, value=values)
                )])
                
                fig_sankey.update_layout(
                    title=f"{fn}: Clustering Flow Across Thresholds",
                    font_size=10, height=600, width=1200
                )
                
                # Save as interactive HTML
                fig_sankey.write_html(file_04)
                print(f"  ✓ Generated: {file_04.name}")
            except Exception as e:
                print(f"  ✗ Failed: {file_04.name} - {str(e)}")
        else:
            print(f"  ⊙ Skipped (exists): {file_04.name}")
        
        if not skip_05:
            try:
                # Plot brain partitions for each threshold
                n_thresh = len(thresholds_sankey)
                fig_brains, axes_brains = plt.subplots(n_thresh, 1, figsize=(10, 4*n_thresh))
                if n_thresh == 1:
                    axes_brains = [axes_brains]
                
                for i, thresh in enumerate(thresholds_sankey):
                    clusters = cluster_assignments_sankey[i]
                    node_colors_level = [cluster_colors_dict_sankey[i][nc] for nc in clusters]
                    
                    node_colors_all_level = node_colors_level.copy()
                    for idx in reversed(indices_to_insert):
                        node_colors_all_level.insert(idx, 'white')
                    
                    cmap_level = ListedColormap(node_colors_all_level)
                    n_clusters = len(np.unique(clusters))
                    
                    plotting.plot_stat_map(atlas_img, bg_img=template, axes=axes_brains[i], 
                                          cmap=cmap_level, colorbar=False,
                                          title=f'Threshold={thresh:.2e}, {n_clusters} clusters')
                
                plt.savefig(file_05, dpi=300, bbox_inches='tight')
                plt.close()
                print(f"  ✓ Generated: {file_05.name}")
            except Exception as e:
                plt.close('all')
                print(f"  ✗ Failed: {file_05.name} - {str(e)}")
        else:
            print(f"  ⊙ Skipped (exists): {file_05.name}")
    except Exception as e:
        print(f"  ✗ Error in Sankey/multithreshold analysis: {str(e)}")
        if not skip_04:
            print(f"  ✗ Failed: {file_04.name}")
        if not skip_05:
            print(f"  ✗ Failed: {file_05.name}")
    
    # ========== 5. SAVE HIERARCHICAL CLUSTERING DATA STRUCTURES ==========
    if not skip_06:
        try:
            # Build the clean hierarchical clustering data for this specific analysis
            thresholds_data = thresholds_sankey
            
            # Create clean data structure for this specific correlation matrix
            clustering_hierarchy_rois_clean_fn = {}
            clustering_hierarchy_indices_clean_fn = {}
            
            for i, thresh in enumerate(thresholds_data):
                clusters = cluster_assignments_sankey[i]
                
                # Create dictionary with node indices
                cluster_dict_indices = {}
                for node_idx, cluster_id in enumerate(clusters):
                    cluster_id_int = int(cluster_id)
                    if cluster_id_int not in cluster_dict_indices:
                        cluster_dict_indices[cluster_id_int] = []
                    cluster_dict_indices[cluster_id_int].append(int(node_idx))
                
                clustering_hierarchy_indices_clean_fn[f'threshold_{i}'] = {
                    'threshold_value': float(thresh),
                    'n_clusters': len(cluster_dict_indices),
                    'clusters': cluster_dict_indices
                }
                
                # Create dictionary with ROI names
                node_idx_to_roi_fn = {}
                roi_counter = 0
                for atlas_idx in range(len(rois)):
                    if atlas_idx not in indices_to_insert:
                        roi_name = rois[atlas_idx]
                        if isinstance(roi_name, bytes):
                            roi_name = roi_name.decode('utf-8')
                        elif isinstance(roi_name, np.bytes_):
                            roi_name = roi_name.decode('utf-8')
                        else:
                            roi_name = str(roi_name)
                        node_idx_to_roi_fn[roi_counter] = roi_name
                        roi_counter += 1
                
                cluster_dict_rois = {}
                for node_idx, cluster_id in enumerate(clusters):
                    cluster_id_int = int(cluster_id)
                    if cluster_id_int not in cluster_dict_rois:
                        cluster_dict_rois[cluster_id_int] = []
                    roi_name = node_idx_to_roi_fn.get(node_idx, f"Unknown_Node_{node_idx}")
                    cluster_dict_rois[cluster_id_int].append(roi_name)
                
                clustering_hierarchy_rois_clean_fn[f'threshold_{i}'] = {
                    'threshold_value': float(thresh),
                    'n_clusters': len(cluster_dict_rois),
                    'clusters': cluster_dict_rois
                }
            
            # Save as JSON (human-readable)
            with open(file_06_json, 'w') as f:
                json.dump({
                    'metadata': {
                        'analysis_label': fn,
                        'subject_id': subject_id,
                        'contrast_folder': contrast_folder,
                        'atlas': 'Schaefer2018_100Parcels_7Networks',
                        'n_roi_total': len(rois),
                        'n_roi_analyzed': len(G_pos.nodes()),
                        'n_roi_lost': len(indices_to_insert),
                        'n_threshold_levels': len(thresholds_data),
                        'threshold_range': [float(thresholds_data[0]), float(thresholds_data[-1])]
                    },
                    'roi_indices': clustering_hierarchy_indices_clean_fn,
                    'roi_names': clustering_hierarchy_rois_clean_fn
                }, f, indent=2)
            
            # Also save as pickle (for easy Python loading)
            with open(file_06_pkl, 'wb') as f:
                pk.dump({
                    'metadata': {
                        'analysis_label': fn,
                        'subject_id': subject_id,
                        'contrast_folder': contrast_folder,
                        'atlas': 'Schaefer2018_100Parcels_7Networks',
                        'n_roi_total': len(rois),
                        'n_roi_analyzed': len(G_pos.nodes()),
                        'n_roi_lost': len(indices_to_insert),
                        'lost_roi_indices': indices_to_insert,
                        'n_threshold_levels': len(thresholds_data),
                        'threshold_range': [float(thresholds_data[0]), float(thresholds_data[-1])]
                    },
                    'roi_indices': clustering_hierarchy_indices_clean_fn,
                    'roi_names': clustering_hierarchy_rois_clean_fn
                }, f)
            
            print(f"  ✓ Generated: {file_06_json.name}")
            print(f"  ✓ Generated: {file_06_pkl.name}")
            print(f"  - Clusters at thresholds: {[len(np.unique(cluster_assignments_sankey[i])) for i in range(len(thresholds_sankey))]}")
        except Exception as e:
            print(f"  ✗ Failed: {file_06_json.name} & {file_06_pkl.name} - {str(e)}")
    else:
        print(f"  ⊙ Skipped (exists): {file_06_json.name} & {file_06_pkl.name}")


SKIPPING (already processed): sub-00027599_0_17Networks_kwCBF4D_s4
  User: sub-00027599, Contrast: 0
  All output files exist (7 files)

SKIPPING (already processed): sub-00027599_0_17Networks_kwCBF4D_s5
  User: sub-00027599, Contrast: 0
  All output files exist (7 files)

SKIPPING (already processed): sub-00027599_0_17Networks_kwCBF4D_sstar
  User: sub-00027599, Contrast: 0
  All output files exist (7 files)

SKIPPING (already processed): sub-00027599_1_kwfcurN_Vaso_s4
  User: sub-00027599, Contrast: 1
  All output files exist (7 files)

SKIPPING (already processed): sub-00027599_1_kwfcurN_Vaso_s5
  User: sub-00027599, Contrast: 1
  All output files exist (7 files)

SKIPPING (already processed): sub-00027599_1_kwfcurN_Vaso_sstar
  User: sub-00027599, Contrast: 1
  All output files exist (7 files)

SKIPPING (already processed): sub-00027599_2_Bold_s4_120s
  User: sub-00027599, Contrast: 2
  All output files exist (7 files)

SKIPPING (already processed): sub-00027599_2_Bold_s5_120s
  U

/home/opisthofulax/Documents/UniPa/projects/multifun-brain/multifunbrain/analysis/lrglib.py:90: RuntimeWarning:

divide by zero encountered in log

/home/opisthofulax/Documents/UniPa/projects/multifun-brain/multifunbrain/analysis/lrglib.py:90: RuntimeWarning:

invalid value encountered in multiply



  ⊙ Skipped (exists): sub-00045109_1_kwfcurN_Vaso_sstar_02_thresholding_network_entropy.pdf
  ⊙ Skipped (exists): sub-00045109_1_kwfcurN_Vaso_sstar_03_clustering_network_dendro_brain.pdf
  - Target clusters: [100, 50, 25, 12, 6, 3]
  - Actual clusters: [100, 46, 25, 12, 6, 3]
  - Thresholds: ['1.07e-06', '3.24e-06', '4.91e-06', '1.95e-05', '7.77e-05', '3.32e-04']
  ⊙ Skipped (exists): sub-00045109_1_kwfcurN_Vaso_sstar_04_sankey_diagram.html
  ✓ Generated: sub-00045109_1_kwfcurN_Vaso_sstar_05_multithreshold_brain_partitions.pdf
  ✓ Generated: sub-00045109_1_kwfcurN_Vaso_sstar_06_clustering_hierarchy.json
  ✓ Generated: sub-00045109_1_kwfcurN_Vaso_sstar_06_clustering_hierarchy.pkl
  - Clusters at thresholds: [100, 46, 25, 12, 6, 3]

Processing: sub-00045109_2_Bold_s4_120s
  User: sub-00045109, Contrast: 2
  Output: /home/opisthofulax/Documents/UniPa/projects/multifun-brain/data/figures/25-11-05/freq-user/sub-00045109/2
  Files to generate: 6/7
  ✓ Generated: sub-00045109_1_kwfcurN_Vaso_s

/home/opisthofulax/Documents/UniPa/projects/multifun-brain/multifunbrain/analysis/lrglib.py:90: RuntimeWarning:

divide by zero encountered in log

/home/opisthofulax/Documents/UniPa/projects/multifun-brain/multifunbrain/analysis/lrglib.py:90: RuntimeWarning:

invalid value encountered in multiply



  ✓ Generated: sub-00045109_2_Bold_s4_120s_02_thresholding_network_entropy.pdf
  ✓ Generated: sub-00045109_2_Bold_s4_120s_03_clustering_network_dendro_brain.pdf
  - Target clusters: [100, 50, 25, 12, 6, 3]
  - Actual clusters: [100, 50, 25, 13, 6, 3]
  - Thresholds: ['3.57e-13', '1.31e-12', '3.10e-12', '1.51e-11', '5.53e-11', '3.12e-08']
  ✓ Generated: sub-00045109_2_Bold_s4_120s_04_sankey_diagram.html
  ✓ Generated: sub-00045109_2_Bold_s4_120s_03_clustering_network_dendro_brain.pdf
  - Target clusters: [100, 50, 25, 12, 6, 3]
  - Actual clusters: [100, 50, 25, 13, 6, 3]
  - Thresholds: ['3.57e-13', '1.31e-12', '3.10e-12', '1.51e-11', '5.53e-11', '3.12e-08']
  ✓ Generated: sub-00045109_2_Bold_s4_120s_04_sankey_diagram.html
  ✓ Generated: sub-00045109_2_Bold_s4_120s_05_multithreshold_brain_partitions.pdf
  ✓ Generated: sub-00045109_2_Bold_s4_120s_06_clustering_hierarchy.json
  ✓ Generated: sub-00045109_2_Bold_s4_120s_06_clustering_hierarchy.pkl
  - Clusters at thresholds: [100, 50, 25, 

...

/home/opisthofulax/Documents/UniPa/projects/multifun-brain/multifunbrain/analysis/lrglib.py:90: RuntimeWarning:

divide by zero encountered in log

/home/opisthofulax/Documents/UniPa/projects/multifun-brain/multifunbrain/analysis/lrglib.py:90: RuntimeWarning:

invalid value encountered in multiply



  ✓ Generated: sub-00045109_3_kwoptcomMIRDenoised_bold_s5_02_thresholding_network_entropy.pdf
  ✓ Generated: sub-00045109_3_kwoptcomMIRDenoised_bold_s5_03_clustering_network_dendro_brain.pdf
  - Target clusters: [100, 50, 25, 12, 6, 3]
  - Actual clusters: [100, 47, 26, 12, 6, 3]
  - Thresholds: ['3.70e-06', '9.51e-06', '1.22e-05', '2.29e-05', '1.71e-04', '5.31e-04']
  ✓ Generated: sub-00045109_3_kwoptcomMIRDenoised_bold_s5_04_sankey_diagram.html
  ✓ Generated: sub-00045109_3_kwoptcomMIRDenoised_bold_s5_03_clustering_network_dendro_brain.pdf
  - Target clusters: [100, 50, 25, 12, 6, 3]
  - Actual clusters: [100, 47, 26, 12, 6, 3]
  - Thresholds: ['3.70e-06', '9.51e-06', '1.22e-05', '2.29e-05', '1.71e-04', '5.31e-04']
  ✓ Generated: sub-00045109_3_kwoptcomMIRDenoised_bold_s5_04_sankey_diagram.html
  ✓ Generated: sub-00045109_3_kwoptcomMIRDenoised_bold_s5_05_multithreshold_brain_partitions.pdf
  ✓ Generated: sub-00045109_3_kwoptcomMIRDenoised_bold_s5_06_clustering_hierarchy.json
  ✓ Gene

/home/opisthofulax/Documents/UniPa/projects/multifun-brain/multifunbrain/analysis/lrglib.py:90: RuntimeWarning:

divide by zero encountered in log

/home/opisthofulax/Documents/UniPa/projects/multifun-brain/multifunbrain/analysis/lrglib.py:90: RuntimeWarning:

invalid value encountered in multiply



  ✓ Generated: sub-00045109_3_kwoptcomMIRDenoised_bold_sstar_02_thresholding_network_entropy.pdf


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f5671a7d010>>
Traceback (most recent call last):
  File "/home/opisthofulax/anaconda3/envs/multifun-brain/lib/python3.13/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 
